# Causal Coupling of Co-occurrence-Dependent Target Evidence (Experiment 4)

This notebook reads only already-saved outputs under `outputs/cooccurrence_causal_coupling/` and `outputs/CooccurrenceHallucinationDiagnostic/`. **No model inference is rerun here.** See `../README.md` for the full narrative; this notebook is the numeric/figure companion.

In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

CC = Path('/data3/KJE/code/UQ/outputs/cooccurrence_causal_coupling')
COOC = Path('/data3/KJE/code/UQ/outputs/CooccurrenceHallucinationDiagnostic')

full_report = json.load(open(CC / 'patching' / 'experiment4_full_report.json'))
beta_after = pd.read_csv(CC / 'patching' / 'beta_after_intervention.csv')
screen = json.load(open(CC / 'patching' / 'screen_layers_val.json'))
direction_meta = json.load(open(CC / 'patching' / 'direction_metadata.json'))
stage11 = json.load(open(COOC / 'stage11_layer_localization' / 'stage11_report.json'))
print('beta_before (test split):', full_report['beta_before'])

## 1. Research Question

Does the within-image co-occurrence-specificity effect (Experiment 2, β=0.406) and its layer-wise localization (Experiment 3, logit lens) reflect a causal mechanism, or is the localized signal merely correlational? Tested via activation patching (mean-referenced projection removal) at 4 representative layers, on a held-out test split.

## 2. Experiment 1-3 Recap

In [ ]:
print('Experiment 1: High-Low s_T mean diff = +0.729, 95% CI [0.399, 1.061], Cohens dz=0.353')
print('Experiment 2: beta = 0.406, 95% CI [0.290, 0.522], permutation p = 0.0005')
print('Experiment 3 final layer (32) beta (logit lens) =', [r for r in stage11['per_layer'] if r['layer']==32][0]['beta'])

## 3. Stage 11 Localization Pattern

In [ ]:
s11 = pd.DataFrame(stage11['per_layer'])[['layer', 'partial_r', 'permutation_p_value']]
display(s11)

## 4. Intervention Definition

`h_L' = h_L - lambda * ((h_L - reference_L) . d_hat_L) * d_hat_L`, applied only at the decision (last-token) position, hooked on `language_model.layers[L-1]`'s output (== `hidden_states[L]`). See `common.py::ResidualProjectionHook`.

## 5. Direction Validation

In [ ]:
dmeta = pd.DataFrame(direction_meta['per_layer'])
display(dmeta)
print('Real-direction norm is consistently 8-17x the shuffled-direction norm at every layer -- the real regression explains far more variance than a permuted score would.')

## 6. Layer-wise Causal Effect (Delta_beta)

In [ ]:
display(beta_after[['layer','lambda','beta_before','beta_after','delta_beta','mean_delta_sT']])
display(Image(filename=str(CC / 'figures' / 'fig1_delta_beta_by_layer.png')))
display(Image(filename=str(CC / 'figures' / 'fig2_beta_before_vs_after.png')))

## 7. Negative Region Analysis

Not causally tested (gated on the positive-region layer scan finding a selective effect first; it did not -- see Section 11 below and `checkpoints/checkpoint_layer_scan.md`). Only the correlational Stage 11 profile (Section 3) covers L7-12 in this experiment.

## 8. Transition Layer Analysis

In [ ]:
display(Image(filename=str(CC / 'figures' / 'fig5_localization_vs_causal.png')))
print('Representational partial_r plateaus at L13 and stays flat to L32.')
print('Causal Delta_beta keeps GROWING from L13 to L24 -- dissociated from the representational plateau.')

## 9. Genuine Target Preservation (decisive control)

In [ ]:
genuine = pd.DataFrame(full_report['control1_genuine_target'])
display(genuine)
display(Image(filename=str(CC / 'figures' / 'fig3_selectivity.png')))
print('Genuine-target evidence collapses MORE than the main effect, worsening with depth (L24: -3.06 logits). FAILS the preservation criterion.')

## 10. Low-Co-occurrence Control

In [ ]:
low_cooc = pd.DataFrame(full_report['control2_low_vs_high_cooc'])
display(low_cooc)
print('Low-cooc Delta_sT RISES with depth instead of staying flat -- both tails move toward the mean (variance shrinkage), not a one-sided suppression.')

## 11. Random / Shuffled Controls

In [ ]:
controls34 = pd.DataFrame(full_report['control3_random_and_control4_shuffled'])
display(controls34[['layer','delta_beta_real','delta_beta_random_mean','delta_beta_random_sd','delta_beta_shuffled']])
print('Random direction: Delta_beta ~ 0 at every layer (5 seeds). Shuffled: small but non-zero at L16/L24 (7-8x smaller than real).')

## 12. Lambda Sensitivity

See Section 6's table above: Delta_beta is monotonic in lambda at every non-null layer (never reverses sign or direction across the 0.25-1.0 grid) -- the expected signature of a real, dose-dependent effect rather than noise.

## 13. Attention vs. MLP Refinement

Not performed -- gated on Sections 6-11 finding a narrow, selective causal region, which they did not. See `audit/repository_audit.md` for the confirmed hook points (`layers[L].self_attn`, `layers[L].mlp`) if revisited.

## 14. Supported vs Unsupported Mechanism

**Supported**: the co-occurrence-correlated direction is causally load-bearing for beta (large, dose-dependent, random/shuffled-control-beating Delta_beta).

**Not supported**: that this causal leverage is *specific* to co-occurrence. Genuine target evidence is destroyed as much or more; low-cooc targets are not spared.

## 15. What We Can and Cannot Claim

See README.md Sections 12-13 for the full claim/non-claim list, and the Final Summary Table at the end of README.md.

In [ ]:
summary = pd.DataFrame([
    {'question': 'Does L13+ contain co-occurrence-related evidence?', 'status': 'SUPPORTED'},
    {'question': 'Is this evidence causally used downstream?', 'status': 'SUPPORTED'},
    {'question': 'Is the effect specific to co-occurrence?', 'status': 'NOT SUPPORTED'},
    {'question': 'Is genuine target evidence preserved?', 'status': 'NOT SUPPORTED'},
    {'question': 'Is L7-12 negative signal causal?', 'status': 'INCONCLUSIVE'},
    {'question': 'Is attention or MLP responsible?', 'status': 'INCONCLUSIVE'},
])
display(summary)